https://www.cnblogs.com/tian777/p/17533477.html

In [1]:
class Validator:

    def __init__(self):
        self.data = {}

    def __get__(self, obj, objtype=None):
        return self.data[obj]

    def __set__(self, obj, value):
        # 校验通过后再赋值
        self.validate(value)
        self.data[obj] = value

    def validate(self, value):
        pass


class Number(Validator):

    def __init__(self, minvalue=None, maxvalue=None):
        super(Number, self).__init__()
        self.minvalue = minvalue
        self.maxvalue = maxvalue

    def validate(self, value):
        if not isinstance(value, (int, float)):
            raise TypeError(f'Expected {value!r} to be an int or float')
        if self.minvalue is not None and value < self.minvalue:
            raise ValueError(
                f'Expected {value!r} to be at least {self.minvalue!r}'
            )
        if self.maxvalue is not None and value > self.maxvalue:
            raise ValueError(
                f'Expected {value!r} to be no more than {self.maxvalue!r}'
            )


class String(Validator):

    def __init__(self, minsize=None, maxsize=None):
        super(String, self).__init__()
        self.minsize = minsize
        self.maxsize = maxsize

    def validate(self, value):
        if not isinstance(value, str):
            raise TypeError(f'Expected {value!r} to be an str')
        if self.minsize is not None and len(value) < self.minsize:
            raise ValueError(
                f'Expected {value!r} to be no smaller than {self.minsize!r}'
            )
        if self.maxsize is not None and len(value) > self.maxsize:
            raise ValueError(
                f'Expected {value!r} to be no bigger than {self.maxsize!r}'
            )


class Person:
    # 定义属性的校验规则 内部用描述符实现
    name = String(minsize=3, maxsize=10)
    age = Number(minvalue=1, maxvalue=120)

    def __init__(self, name, age):
        self.name = name
        self.age = age


# 属性符合规则
p1 = Person('zhangsan', 20)
print(p1.name, p1.age)

# 属性不符合规则
p2 = Person('a', 20)
# ValueError: Expected 'a' to be no smaller than 3
p3 = Person('zhangsan', -1)
# ValueError: Expected -1 to be at least 1



zhangsan 20


ValueError: Expected 'a' to be no smaller than 3

In [3]:
# 类型检查的描述器

class Typed:
    # 变量名称，数据类型
    def __init__(self, name, expected_type):
        self.name = name
        self.expected_type = expected_type

    # 获取字典里的变量名称
    def __get__(self, instance, cls):
        if instance is None:
            return self
        else:
            return instance.__dict__[self.name]

    # 设置变量，如果不符合期望类型抛出异常
    def __set__(self, instance, value):
        if not isinstance(value, self.expected_type):
            raise TypeError('Expected ' + str(self.expected_type))
        instance.__dict__[self.name] = value

    # 删除变量
    def __delete__(self, instance):
        del instance.__dict__[self.name]


# Class decorator that applies it to selected attributes
# 类型检查的描述器
def typeassert(**kwargs):
    # {'name': <class 'str'>, 'shares': <class 'int'>, 'price': <class 'float'>}
    def decorate(cls):
        for name, expected_type in kwargs.items():
            # Attach a Typed descriptor to the class
            # 给这个类添加一个类型描述器
            setattr(cls, name, Typed(name, expected_type))
        return cls

    return decorate


# 描述器应用样例
@typeassert(name=str, age=int)
class People:
    def __init__(self, name, age):
        self.name = name
        self.age = age


people = People('xiaoxin', 18)
print(people.__dict__)
people = People('xiaoxin', 18.6)
print(people.__dict__)




{'name': 'xiaoxin', 'age': 18}


TypeError: Expected <class 'int'>